In [4]:
# 2025.12.01 Ensemble Set No 4 돌리기

In [5]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [16]:
import os

import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics         import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics         import classification_report
from sklearn.metrics         import roc_auc_score
from sklearn.metrics         import precision_recall_curve, classification_report
from sklearn.datasets        import make_classification
from sklearn.preprocessing   import RobustScaler

# Model import
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import StackingClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.ensemble       import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.linear_model   import LinearRegression
from sklearn.linear_model   import SGDClassifier
from sklearn.svm            import SVC
from sklearn.svm            import LinearSVC
from xgboost                import XGBClassifier
from xgboost                import plot_importance
from lightgbm               import LGBMClassifier
from catboost               import CatBoostClassifier
from sklearn.ensemble       import HistGradientBoostingClassifier
from sklearn.calibration    import CalibratedClassifierCV # stacking시 LSVC 에러용

# hyperopt 용
from hyperopt               import hp

# 사용자 Functions import
import HyperParams          as HP 
import utils.data_sampling  as ds 

import importlib

from utils import user_utils
importlib.reload(user_utils)

from utils import user_utils    as uu
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import model_utils   as mu
from utils import modeling      as mo

from utils.hyperopt_search import hyperopt_search, train_and_evaluate

In [7]:
# -------------------------------------------------------
# 🔷 모델 생성 함수 (HyperOpt 파라미터 자동 적용)
# -------------------------------------------------------

def get_models():
    models = {
        "cat": CatBoostClassifier(**HP.cb_best_params),
        "dt": DecisionTreeClassifier(**HP.dt_basic_params),
        "gb": GradientBoostingClassifier(**HP.gb_best_params),
        "hgb": HistGradientBoostingClassifier(**HP.hgb_best_params),
        "lgbm": LGBMClassifier(**HP.lgbm_best_param2),
        "lr": LogisticRegression(**HP.lr_best_params),
        "lir" : LinearRegression(**HP.lir_best_params),
        "lsvc": LinearSVC(**HP.lsvc_best_params),
        "mlp": MLPClassifier(**HP.mlp_basic_params),
        "rf": RandomForestClassifier(**HP.rf_best_params),
        "sgd": SGDClassifier(**HP.sgd_best_params),
        "svm_rbf": SVC(**HP.svc_rbf_best_params),
        "xgb": XGBClassifier(**HP.xgb_best_params, eval_metric='logloss'),
    }

    return models
# eof ----------------------------------------------------------------

In [8]:
# 결과받을 딕셔너리
results = {}
team_rs = 23 # 우리팀 random_state

In [9]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [10]:
# 2. 데이터 전처리
# 2.1 Time 컬럼 삭제 , 데이터,타겟 분리
X_features, y_target = pp.split_features_target(raw_df, cols= 'Time')
X_features.shape, y_target.shape

((284807, 29), (284807,))

In [11]:
# 2.2 이상치를 경계값으로 치환
cap_X_feature = pp.cap_outliers(X_features)

In [12]:
# 2.3 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = pp.data_split(cap_X_feature, y_target)

In [12]:
# 학습/검증 데이터 분리
# X_tr, X_val, y_tr, y_val = pp.data_split(X_train, y_train, size=0.4)

In [ ]:
# 2.4 Over Sampling 하는 경우
X_over, y_over = ds.oversampling_smote(X_train, y_train)


In [14]:
# Over Sampling한 경우 학습/검증 데이터 분리
# X_tr_over, X_val_over, y_tr_over, y_val_over = pp.data_split(X_over, y_over, size=0.4)

In [15]:
# 사용자 Functions import - 에러나서 다시 실행
# import importlib
# from utils import hyperopt_search
# importlib.reload(hyperopt_search)

# from utils.hyperopt_search import hyperopt_search, train_and_evaluate

In [16]:
# BestOpt 찾고 나서 스케일적용 버전 만들어서 모델링하기 
# 2.5 StandardScaler 적용
X_train_sscaled, X_test_sscaled, scaler = pp.scale_data(X_train, X_test)


In [17]:
# 2.6 robustScaler 적용

rscaler = RobustScaler()
X_train_rscaled = rscaler.fit_transform(X_train)   # 학습 데이터로 fit + transform
X_test_rscaled = rscaler.transform(X_test)         # 테스트 데이터는 transform만


In [13]:
def create_lr(best_params):
    """
    LogisticRegression 모델을 HyperOpt/Optuna로 찾은 best_params 기반으로 생성하는 함수.

    이 함수는 LogisticRegression의 solver와 penalty 조합이 유효한지 검사하고,
    유효하지 않은 조합이 들어올 경우 자동으로 수정하여 안전하게 모델을 생성한다.

    Parameters
    ----------
    best_params : dict
        HyperOpt 또는 Optuna로 최적화한 LogisticRegression의 최적 파라미터 딕셔너리.
        예: {"C": 0.1, "solver": "liblinear", "penalty": "l1"}

    Returns
    -------
    LogisticRegression
        최종적으로 검증된 파라미터로 생성된 LogisticRegression 모델.

    Notes
    -----
    - solver에 따라 사용할 수 있는 penalty 종류가 다르기 때문에,
      최적화 결과가 잘못된 조합을 반환할 가능성이 있음.
    - 안전성을 위해 직접 검증한 후 잘못된 penalty는 'l2'로 자동 변경한다.
    - 변경이 발생하면 경고 메시지를 출력한다.
    """

    # 최적화된 solver, penalty 값을 가져오고 기본값 설정
    solver = best_params.get('solver', 'liblinear')
    penalty = best_params.get('penalty', 'l2')

    # solver별로 허용되는 penalty 목록 정의
    valid_penalties = {
        'liblinear': ['l1', 'l2'],
        'lbfgs': ['l2', 'none'],
        'saga': ['l1', 'l2', 'elasticnet', 'none'],
        'newton-cg': ['l2', 'none'],
    }

    # penalty가 solver에 맞지 않으면 자동 수정
    if penalty not in valid_penalties.get(solver, []):
        print(f"[WARN] penalty '{penalty}' is incompatible with solver '{solver}'. Using 'l2'")
        best_params['penalty'] = 'l2'

    # 유효한 파라미터로 LogisticRegression 모델 생성
    return LogisticRegression(**best_params)
# eof -----------------------------------------------------------


In [ ]:
# ======================================
# 🔷 Stacking #1: CatBoost + XGB + LGBM
# ======================================

models = get_models()

estimators_1 = [
    ('cat', models['cat']),
    ('xgb', models['xgb']),
    ('lgbm', models['lgbm']),
]


stack_1 = StackingClassifier(
    estimators=estimators_1,
    final_estimator=create_lr(HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
option_name = 'cat+xgb+lgbm_ho_best'
# results = uu.get_model_train_eval(stack_1, f'{option_name}', X_train, X_test, y_train, y_test)


In [ ]:
# ======================================
# 🔷 Stacking #2: LGBM + RF + MLP
# ======================================

models = get_models()

estimators_2 = [
    ('lgbm', models['lgbm']),
    ('rf', models['rf']),
    ('mlp', models['mlp']),
]

stack_2 = StackingClassifier(
    estimators=estimators_2,
    final_estimator=create_lr(HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
option_name = 'lgbm+rf+mlp_ho_best'
# results = uu.get_model_train_eval(stack_2, f'{option_name}', X_train, X_test, y_train, y_test)


In [ ]:
# ======================================
# 🔷 Stacking #3: CatBoost + GB + SVM(RBF)
# ======================================
models = get_models()
estimators_3 = [
    ('cat', models['cat']),
    ('gb', models['gb']),
    ('svm_rbf', models['svm_rbf']),
]

stack_3 = StackingClassifier(
    estimators=estimators_3,
    final_estimator=create_lr(HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
option_name = 'cat+gb+svm_rbf_ho_best'
# results = uu.get_model_train_eval(stack_3, f'{option_name}', X_train, X_test, y_train, y_test)

In [14]:
def create_calibrated_lsvc(model):
    """
    predict_proba를 지원하는 LinearSVC 생성
    
    CalibratedClassifierCV는 decision_function을 확률로 변환해줍니다.
    """
    base_svc = model
    calibrated_svc = CalibratedClassifierCV(base_svc, cv=3)
    return calibrated_svc

In [39]:
from utils import user_utils
importlib.reload(user_utils)

from utils import user_utils    as uu

In [ ]:
# ======================================
# 🔷 Stacking #4: XGB + LR + SVM(linear)
# ======================================
# LR은 메타모델로 사용하는 게 더 좋아서 base에는 넣지 않음
models = get_models()
est_lr = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='liblinear',
    class_weight='balanced'
)

estimators_4 = [
    ('xgb', models['xgb']),
    ('lsvc', create_calibrated_lsvc(models['lsvc'])),
]

stack_4 = StackingClassifier(
    estimators=estimators_4,
    final_estimator=est_lr,
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
option_name = 'stack4(xgb+lsvc+lr)_ho_best'
results = uu.get_model_train_eval(stack_4, f'{option_name}', X_train, X_test, y_train, y_test)


✓ 모델 저장 완료: ../models\stack4(xgb+lsvc+lr)_ho_best.pkl
  파일 크기: 0.17 MB
folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'result_dict': {'AUC': 0.9734, '정확도': 0.988, '정밀도': 0.1138, '재현율': 0.8776, 'F1': 0.2014, 'F2': 0.3746}, '오차행렬': [[56194, 670], [12, 86]], '실행 시간': 21.7402}


In [42]:
results


{'result_dict': {'AUC': 0.9734,
  '정밀도': 0.1138,
  '재현율': 0.8776,
  'F1': 0.2014,
  'F2': 0.3746,
  '실행시간': 21.7402,
  '하이퍼파라미터': 'None'},
 'Base Estimators': {'xgb': {'정밀도': 0.4256, '재현율': 0.8469, 'F1': 0.5666},
  'lsvc': {'정밀도': 0.7297, '재현율': 0.551, 'F1': 0.6279}}}

In [ ]:
# ======================================
# 🔷 Stacking #5: RF + SGD + MLP + LR
# ======================================

estimators_5 = [
    ('rf', models['rf']),
    ('sgd', models['sgd']),
    ('mlp', models['mlp']),
]

stack_5 = StackingClassifier(
    estimators=estimators_5,
    # final_estimator=LogisticRegression(class_weight='balanced', solver='liblinear'),
    final_estimator=create_lr(HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)

In [ ]:
# ===========================================
# 🔷 5개 Stack 모델을 Soft Voting으로 결합
# ===========================================

from sklearn.ensemble import VotingClassifier

# VotingClassifier는 원래 predict_proba를 지원하는 모델만 가능
# 우리 스택 모델들은 모두 predict_proba 지원하므로 문제 없음
weights_desc = '''
| 스택                         | 추천 이유     | weight |
| -------------------------- | --------- | ------ |
| Stack #1 (Boosting+LR)     | 대부분 최고 성능 | **3**  |
| Stack #3 (CatBoost+GB+SVM) | Recall 강함 | **2**  |
| Stack #2 (LGBM+RF+MLP)     | 안정적       | **2**  |
| Stack #4 (XGB+LR+SVM)      | 선형 경계 보정  | **1**  |
| Stack #5 (RF+SGD+MLP)      | 편향 다양성 확보 | **1**  |
'''

voting_ensemble = VotingClassifier(
    estimators=[
        ('stack1', stack_1),
        ('stack2', stack_2),
        ('stack3', stack_3),
        ('stack4', stack_4),
        ('stack5', stack_5),
    ],
    voting='soft',          # 🔥 중요: 확률 기반 soft voting
    weights=[3, 2, 2, 1, 1], # 가중치 
    n_jobs=-1
)

In [ ]:
# 시각화
mo.model_metrics_graph(results, 'LGBM 데이터별 성능지표 비교')